In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import os
import sys
from typing import Dict, List, Optional, Sequence, Tuple
import pandas as pd
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
# Add src to path
ROOT = "/Users/liuxi/Desktop/RFA_GNN"
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import importlib
import data_loader
importlib.reload(data_loader)
from sklearn.preprocessing import LabelEncoder
from base_gnn import BaseLineGAT
from data_loader import load_rfa_data, build_combined_gnn
from train_deepcop import cold_split
from xpert_preprocess import generate_cleaned_gctx_pairs
print(data_loader.__file__)

/Users/liuxi/Desktop/RFA_GNN/src/data_loader.py


In [12]:
debug = True
limit = 2000
epochs = 2

tf_path=os.path.join(ROOT, "data/omnipath/omnipath_tf_regulons.csv")
ppi_path=os.path.join(ROOT, "data/omnipath/omnipath_interactions.csv")
landmark_path=os.path.join(ROOT, "data/landmark_genes.json")
with open(landmark_path, "r") as f:
    _lm = json.load(f)
if isinstance(_lm, dict) and "landmark_genes" in _lm:
    _lm_list = _lm["landmark_genes"]
elif isinstance(_lm, list):
    _lm_list = _lm
else:
    _lm_list = []
landmark_genes = [str(g.get("entrez_id") or g.get("pr_gene_id")) for g in _lm_list if isinstance(g, dict) and (g.get("entrez_id") or g.get("pr_gene_id"))]
landmark_genes = [g for g in landmark_genes if g and g != "None"]
full_gene_path=os.path.join(ROOT, "data/GSE92742_Broad_LINCS_gene_info.txt")
string_path = os.path.join(ROOT, "data/string_interactions_mapped.csv")
confid_threshold= 0.9
siginfo_path = os.path.join(ROOT, "data/siginfo_beta.txt")
ctl_csv_path = os.path.join(ROOT, "data/cmap/level3_beta_ctl_n188708x12328.h5")
trt_csv_path = os.path.join(ROOT, "data/cmap/level3_beta_trt_cp_n1805898x12328.h5")


In [11]:
adj_matrix, node_list, gene2idx, edge_index = build_combined_gnn( tf_path=tf_path,
                    ppi_path=ppi_path,
                    landmark_path=landmark_path,
                    landmark_genes=landmark_genes,
                    full_gene_path=full_gene_path,
                    string_path=string_path,
                    confid_threshold=confid_threshold,
                    directed=False)


>>> 正在构建 Combined GNN (TF + PPI) ...
正在加载全基因映射: ../data/GSE92742_Broad_LINCS_gene_info.txt
加载 TF 调控网络: ../data/omnipath/omnipath_tf_regulons.csv
TF 边数: 49650
加载 PPI 网络: ../data/omnipath/omnipath_interactions.csv
PPI 边数: 26633
加载 STRING 网络: ../data/string_interactions_mapped.csv
STRING 添加的边数: 51058
  STRING 边数: 51058
Combined Graph 构建完成: 10786 节点, 92543 边 (含权重)


In [ ]:
data = load_rfa_data(
    ctl_csv_path,
    trt_csv_path,
    landmark_path=landmark_path,
    siginfo_path=siginfo_path,
    use_landmark_genes=True,
    full_gene_path=full_gene_path,
    max_samples=limit if debug else 20000,
)
if data is None:
    raise RuntimeError("load_rfa_data returned None")

In [ ]:
train_data, test_data = cold_split(data)
train_ctl, train_trt, train_drug = train_data
test_ctl, test_trt, test_drug = test_data
le = LabelEncoder()
all_cells = data['cell_names']
cell_idx = le.fit_transform(all_cells)
num_cells = len(le.classes_)
print(f"检测到 {num_cells} 个细胞系。")
# Split
unique_drugs = np.unique(data['drug_ids'])
np.random.seed(42)
test_drugs = np.random.choice(unique_drugs, int(len(unique_drugs) * 0.2), replace=False)

test_mask = np.isin(data['drug_ids'], test_drugs)
train_mask = ~test_mask

train_ctl = data['X_ctl'][train_mask]
train_trt = data['y_delta'][train_mask]
train_drug = data['X_drug'][train_mask]
train_cells = cell_idx[train_mask]

test_ctl = data['X_ctl'][test_mask]
test_trt = data['y_delta'][test_mask]
test_drug = data['X_drug'][test_mask]
test_cells = cell_idx[test_mask]

In [ ]:
num_genes = int(adj_matrix.shape[0])
model = BaseLineGAT(
    num_genes=num_genes,
    num_cells=int(num_cells),
    hidden_dim=64,
    num_heads=4,
    dropout=0.2,
    use_residual=False,
    use_drug_embedding=False,
)
print("BaseLineGAT init ok | num_genes=", num_genes, "num_cells=", int(num_cells))

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=5e-4) # Increased LR
    
    # 自定义 Masked Loss (仅对 Landmark 基因计算 Loss)
loss_mask = tf.constant(data['loss_mask'], dtype=tf.float32)
def pcc_loss(y_true, y_pred):
    mx = tf.reduce_mean(y_true, axis=1, keepdims=True)
    my = tf.reduce_mean(y_pred, axis=1, keepdims=True)
    xm = y_true - mx
    ym = y_pred - my
    r_num = tf.reduce_sum(xm * ym, axis=1)
    r_den = tf.sqrt(tf.reduce_sum(tf.square(xm), axis=1) * tf.reduce_sum(tf.square(ym), axis=1) + 1e-8)
    r = r_num / r_den
    return 1.0 - tf.reduce_mean(r)

def masked_combined_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    mask = tf.cast(loss_mask, tf.float32)
    mse = tf.reduce_sum(tf.square(y_true - y_pred) * mask)
    valid_count = tf.reduce_sum(mask)
    mse = mse / tf.maximum(valid_count, 1.0)
    valid_indices = tf.where(loss_mask[0] > 0)[:, 0]
    yt = tf.gather(y_true, valid_indices, axis=1)
    yp = tf.gather(y_pred, valid_indices, axis=1)
    pcc = pcc_loss(yt, yp)
    return mse + 5.0 * pcc
class GAT_Wrapper(keras.Model):
        def __init__(self, gat_model, adj_matrix):
            super().__init__()
            self.gat = gat_model
            self.adj = tf.constant(adj_matrix, dtype=tf.float32)
            
        def call(self, inputs):
            ctl, drug_targets, cell_idx = inputs
            cell_idx = tf.cast(cell_idx, tf.int32)
            return self.gat([self.adj, ctl, drug_targets, cell_idx])
            
wrapped_model = GAT_Wrapper(model, adj_matrix)
wrapped_model.compile(optimizer=optimizer, loss=masked_combined_loss, metrics=[keras.metrics.MeanSquaredError()])

In [ ]:
def eval_pcc_mse(model, ctl, drug, cells, y_true, loss_mask, batch_size=32, max_eval=None):
    if max_eval is not None and len(ctl) > int(max_eval):
        rng = np.random.default_rng(0)
        idx = rng.choice(len(ctl), size=int(max_eval), replace=False)
        ctl = ctl[idx]
        drug = drug[idx]
        cells = cells[idx]
        y_true = y_true[idx]
    pred = model.predict([ctl, drug, cells], batch_size=batch_size, verbose=0)
    valid_indices = np.where(loss_mask[0] > 0)[0]
    y_true_valid = y_true[:, valid_indices]
    pred_valid = pred[:, valid_indices]
    pcc_list = []
    for i in range(len(y_true_valid)):
        yt = y_true_valid[i]
        yp = pred_valid[i]
        if np.std(yt) > 1e-6 and np.std(yp) > 1e-6:
            p, _ = pearsonr(yt, yp)
            pcc_list.append(p)
    avg_pcc = float(np.mean(pcc_list)) if pcc_list else 0.0
    mse = float(mean_squared_error(y_true_valid, pred_valid))
    return {"mse": mse, "pcc": avg_pcc}


class PCCCallback(keras.callbacks.Callback):
    def __init__(self, loss_mask, train_data, val_data, batch_size=32, max_eval=2048):
        super().__init__()
        self.loss_mask = loss_mask
        self.train_data = train_data
        self.val_data = val_data
        self.batch_size = int(batch_size)
        self.max_eval = int(max_eval) if max_eval is not None else None
        self.train_pcc = []
        self.val_pcc = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        ctl_tr, drug_tr, cell_tr, y_tr = self.train_data
        ctl_va, drug_va, cell_va, y_va = self.val_data
        tr = eval_pcc_mse(self.model, ctl_tr, drug_tr, cell_tr, y_tr, self.loss_mask, batch_size=self.batch_size, max_eval=self.max_eval)
        va = eval_pcc_mse(self.model, ctl_va, drug_va, cell_va, y_va, self.loss_mask, batch_size=self.batch_size, max_eval=self.max_eval)
        self.train_pcc.append(tr["pcc"])
        self.val_pcc.append(va["pcc"])
        logs["pcc"] = tr["pcc"]
        logs["val_pcc"] = va["pcc"]
        print(f"Epoch {epoch+1}: pcc={tr['pcc']:.4f} val_pcc={va['pcc']:.4f}")


pcc_cb = PCCCallback(
    loss_mask=data['loss_mask'],
    train_data=(train_ctl, train_drug, train_cells, train_trt),
    val_data=(test_ctl, test_drug, test_cells, test_trt),
    batch_size=32,
    max_eval=2048,
)

In [ ]:
# 5. Train
print("\n>>> Phase 4: 开始训练 <<<")
history = wrapped_model.fit(
    [train_ctl, train_drug, train_cells],
    train_trt,
    epochs=epochs,
    batch_size=32, 
    callbacks=[pcc_cb],
    validation_data=([test_ctl, test_drug, test_cells], test_trt)
)

In [ ]:
import matplotlib.pyplot as plt

hist = history.history
epochs = np.arange(1, len(hist.get('loss', [])) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(epochs, hist.get('loss', []), label='train_loss')
if 'val_loss' in hist:
    axes[0].plot(epochs, hist.get('val_loss', []), label='val_loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(epochs, hist.get('mean_squared_error', []), label='train_mse')
if 'val_mean_squared_error' in hist:
    axes[1].plot(epochs, hist.get('val_mean_squared_error', []), label='val_mse')
axes[1].set_title('MSE')
axes[1].legend()

pcc_epochs = np.arange(1, len(pcc_cb.train_pcc) + 1)
axes[2].plot(pcc_epochs, pcc_cb.train_pcc, label='train_pcc')
axes[2].plot(pcc_epochs, pcc_cb.val_pcc, label='val_pcc')
axes[2].set_title('Sample-wise PCC (subset)')
axes[2].legend()

plt.show()

In [ ]:
# 6. Eval (Simplified)
print("评估中 (只评估 Landmark 基因)...")
train_metrics = eval_pcc_mse(wrapped_model, train_ctl, train_drug, train_cells, train_trt, data['loss_mask'], batch_size=32, max_eval=20000)
test_metrics = eval_pcc_mse(wrapped_model, test_ctl, test_drug, test_cells, test_trt, data['loss_mask'], batch_size=32, max_eval=None)
print(f"Train | MSE: {train_metrics['mse']:.4f} | Sample-wise PCC: {train_metrics['pcc']:.4f}")
print(f"Test  | MSE: {test_metrics['mse']:.4f} | Sample-wise PCC: {test_metrics['pcc']:.4f}")

pred = wrapped_model.predict([test_ctl, test_drug, test_cells], batch_size=32, verbose=0)
valid_indices = np.where(data['loss_mask'][0] > 0)[0]
test_trt_valid = test_trt[:, valid_indices]
pred_valid = pred[:, valid_indices]
flat_pcc = pearsonr(test_trt_valid.flatten(), pred_valid.flatten())[0]
print(f"Test Global PCC (flatten): {float(flat_pcc):.4f}")